In [ ]:
# --- Environment Setup and Context Initialization ---
import os
import tensorflow as tf
import tensorflow_model_analysis as tfma

from tfx import v1 as tfx
from tfx.types import Channel
from tfx.proto import pusher_pb2, trainer_pb2
from tfx.components import Tuner, Trainer, Transform
from tfx.dsl.components.common.resolver import Resolver
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy

tf.get_logger().propagate = False

print('TensorFlow version: {}'.format(tf.__version__))
print('TFX version: {}'.format(tfx.__version__))

PIPELINE_NAME = 'aldomp7-pipeline'
PIPELINE_ROOT = os.path.join(PIPELINE_NAME, 'pipelines')
METADATA_PATH = os.path.join(PIPELINE_NAME, 'metadata', 'metadata.db')
SERVING_MODEL_DIR = os.path.join(PIPELINE_NAME, 'serving_model')

context = InteractiveContext(pipeline_root=PIPELINE_ROOT, metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(METADATA_PATH))

c:\Users\Alls\venv\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.0). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
c:\Users\Alls\venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
c:\Users\Alls\venv\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with

TensorFlow version: 2.13.0
TFX version: 1.14.0


In [ ]:
# --- Data Ingestion ---
DATA_ROOT = 'data'

example_gen = tfx.components.CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 70
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [ ]:
# --- Data Validation ---
statistics_gen = tfx.components.StatisticsGen(examples=example_gen.outputs['examples'])
context.run(statistics_gen)
context.show(statistics_gen.outputs['statistics'])

schema_gen = tfx.components.SchemaGen(statistics=statistics_gen.outputs['statistics'])
context.run(schema_gen)
context.show(schema_gen.outputs['schema'])

example_validator = tfx.components.ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)
context.show(example_validator.outputs['anomalies'])

,Type,Presence,Valency,Domain
Feature name,,,,
'DEATH_EVENT',INT,required,,-
'age',FLOAT,required,,-
'anaemia',INT,required,,-
'creatinine_phosphokinase',INT,required,,-
'diabetes',INT,required,,-
'ejection_fraction',INT,required,,-
'high_blood_pressure',INT,required,,-
'platelets',FLOAT,required,,-
'serum_creatinine',FLOAT,required,,-


In [ ]:
# --- Data Transformation ---
TRANSFORM_MODULE_FILE = 'modules/heart_disease_transform.py'

transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=TRANSFORM_MODULE_FILE
)

context.run(transform)

INFO:tensorflow:Assets written to: aldomp7-pipeline\pipelines\Transform\transform_graph\74\.temp_path\tftransform_tmp\d8210367588b4e908d7543d0801e1399\assets
INFO:tensorflow:struct2tensor is not available.
INFO:tensorflow:tensorflow_decision_forests is not available.
INFO:tensorflow:tensorflow_text is not available.
INFO:tensorflow:Assets written to: aldomp7-pipeline\pipelines\Transform\transform_graph\74\.temp_path\tftransform_tmp\c5506370cd274842a8f2cefc3b17d625\assets
INFO:tensorflow:struct2tensor is not available.
INFO:tensorflow:tensorflow_decision_forests is not available.
INFO:tensorflow:tensorflow_text is not available.
INFO:tensorflow:struct2tensor is not available.
INFO:tensorflow:tensorflow_decision_forests is not available.
INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 74
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [ ]:
# --- Hyperparameter Tuning ---
TUNER_MODULE_FILE = 'modules/heart_disease_tuner.py'

tuner = Tuner(
    module_file=TUNER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(num_steps=100),
    eval_args=trainer_pb2.EvalArgs(num_steps=50)
)

context.run(tuner)

Trial 20 Complete [00h 00m 03s]
val_binary_accuracy: 0.7777777910232544

Best val_binary_accuracy So Far: 0.8222222328186035
Total elapsed time: 00h 01m 00s
Results summary
Results in aldomp7-pipeline\pipelines\.temp\75\heart_disease_tuning
Showing 10 best trials
Objective(name="val_binary_accuracy", direction="max")

Trial 14 summary
Hyperparameters:
units: 64
dropout: 0.2
learning_rate: 0.01
Score: 0.8222222328186035

Trial 03 summary
Hyperparameters:
units: 64
dropout: 0.5
learning_rate: 0.001
Score: 0.8111110925674438

Trial 04 summary
Hyperparameters:
units: 256
dropout: 0.1
learning_rate: 0.0001
Score: 0.800000011920929

Trial 12 summary
Hyperparameters:
units: 224
dropout: 0.1
learning_rate: 0.01
Score: 0.800000011920929

Trial 09 summary
Hyperparameters:
units: 192
dropout: 0.5
learning_rate: 0.01
Score: 0.7888888716697693

Trial 00 summary
Hyperparameters:
units: 192
dropout: 0.1
learning_rate: 0.001
Score: 0.7777777910232544

Trial 02 summary
Hyperparameters:
units: 160
dropo

ExecutionResult(
    component_id: Tuner
    execution_id: 75
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [ ]:
# --- Model Training ---
TRAINER_MODULE_FILE = 'modules/heart_disease_trainer.py'

trainer = Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(num_steps=100),
    eval_args=trainer_pb2.EvalArgs(num_steps=50)
)

context.run(trainer)

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 age_xf (InputLayer)         [(None, 1)]                  0         []                            
                                                                                                  
 creatinine_phosphokinase_x  [(None, 1)]                  0         []                            
 f (InputLayer)                                                                                   
                                                                                                  
 ejection_fraction_xf (Inpu  [(None, 1)]                  0         []                            
 tLayer)                                                                                          
                                                                                            

ExecutionResult(
    component_id: Trainer
    execution_id: 76
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [ ]:
# --- Evaluation Configuration and Model Resolver ---
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='DEATH_EVENT')],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=['sex'])
    ],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name='ExampleCount'),
            tfma.MetricConfig(
                class_name='BinaryAccuracy',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={'value': 0.5} 
                    ),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={'value': -1e-10}
                    )
                )
            )
        ])
    ]
)

model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('latest_blessed_model_resolver')

context.run(model_resolver)

ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 77
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [ ]:
# --- Model Analysis and Evaluation ---
evaluator = tfx.components.Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)

context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 78
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [ ]:
# --- Model Deployment (Pusher) ---
pusher = tfx.components.Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)

context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 79
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [ ]:
# --- Verify Saved Model ---
print("Model:", os.listdir(SERVING_MODEL_DIR))

Model: ['1771172094']
